Problema 13.......
programa 4 simplificado: construye un agente que juegue un versión pequeña que conecta 4 . explica como evalúa jugadas , explora columnas y explora jugadas ganadas 


In [ ]:
# librerias
import math
import random
import copy

1. Configuración del tablero y funciones del entorno

In [ ]:
# tamaño del tablero y codigos de las piezas
FILAS = 5
COLS = 4
VACIO   = 0
HUMANO  = 1
AGENTE  = 2

def nuevo_tablero():
    """Crea un tablero vacío de 5x4"""
    return [[VACIO] * COLS for _ in range(FILAS)]

def mostrar_tablero(tablero):
    """Imprime el tablero con símbolos"""
    iconos = {VACIO: ' . ', HUMANO: ' X ', AGENTE: ' O '}
    print()
    print("  col: 0   1   2   3")
    print("  " + "-" * 19)
    for i, fila in enumerate(tablero):
        contenido = "".join(iconos[celda] for celda in fila)
        print(f"f{i} |{contenido}|")
    print("  " + "-" * 19)
    print()

def columna_disponible(tablero, col):
    """Verifica si hay espacio en la columna"""
    return tablero[0][col] == VACIO

def fila_libre(tablero, col):
    """Devuelve la fila más baja vacía de esa columna"""
    for f in range(FILAS - 1, -1, -1):
        if tablero[f][col] == VACIO:
            return f
    return -1

def poner_ficha(tablero, col, pieza):
    """Coloca la ficha en la posición correcta"""
    f = fila_libre(tablero, col)
    if f != -1:
        tablero[f][col] = pieza
        return True
    return False

def hay_ganador(tablero, pieza):
    """Revisa si hay 3 en línea (horizontal, vertical o diagonal)"""
    # Horizontal
    for f in range(FILAS):
        for c in range(COLS - 2):
            if tablero[f][c] == tablero[f][c+1] == tablero[f][c+2] == pieza:
                return True
    # Vertical
    for c in range(COLS):
        for f in range(FILAS - 2):
            if tablero[f][c] == tablero[f+1][c] == tablero[f+2][c] == pieza:
                return True
    # Diagonal hacia abajo-derecha
    for f in range(FILAS - 2):
        for c in range(COLS - 2):
            if tablero[f][c] == tablero[f+1][c+1] == tablero[f+2][c+2] == pieza:
                return True
    # Diagonal hacia arriba-derecha
    for f in range(2, FILAS):
        for c in range(COLS - 2):
            if tablero[f][c] == tablero[f-1][c+1] == tablero[f-2][c+2] == pieza:
                return True
    return False

def tablero_lleno(tablero):
    """Devuelve True si ya no hay espacio"""
    return all(tablero[0][c] != VACIO for c in range(COLS))

def columnas_validas(tablero):
    """Lista de columnas donde aún se puede jugar"""
    return [c for c in range(COLS) if columna_disponible(tablero, c)]

print("Entorno listo. Tablero de ejemplo:")
mostrar_tablero(nuevo_tablero())

Entorno listo. Tablero de ejemplo:

  col: 0   1   2   3
  -------------------
f0 | .  .  .  . |
f1 | .  .  .  . |
f2 | .  .  .  . |
f3 | .  .  .  . |
f4 | .  .  .  . |
  -------------------



2. El Agente con UCB

In [ ]:
class AgenteConecta4:




    def __init__(self):
        # veces jugadas en cada colum
        self.conteo = [0] * COLS
        # suma de recomp
        self.ganancia = [0.0] * COLS
        # total de jugads
        self.total_jugadas = 0

    def ucb(self, col):
       
        if self.conteo[col] == 0:
            return float('inf')
        
        promedio   = self.ganancia[col] / self.conteo[col]
        exploracion = math.sqrt(2 * math.log(self.total_jugadas) / self.conteo[col])
        return promedio + exploracion

    def elegir_columna(self, tablero):
       
        validas = columnas_validas(tablero)
        if not validas:
            return None

       
        for col in validas:
            copia = copy.deepcopy(tablero)
            poner_ficha(copia, col, AGENTE)
            if hay_ganador(copia, AGENTE):
                return col

        
        for col in validas:
            copia = copy.deepcopy(tablero)
            poner_ficha(copia, col, HUMANO)
            if hay_ganador(copia, HUMANO):
                return col

       

        mejor_col = max(validas, key=lambda c: self.ucb(c))
        return mejor_col

    def actualizar(self, col, recompensa):
        
        self.conteo[col]   += 1
        self.ganancia[col] += recompensa
        self.total_jugadas += 1

    def mostrar_estadisticas(self):
        
        print("\n--- Estadísticas del agente ---")
        print(f"{'Columna':<10} {'Jugadas':<10} {'Ganancia':<12} {'Promedio':<10}")
        print("-" * 45)
        for c in range(COLS):
            jugadas  = self.conteo[c]
            ganancia = self.ganancia[c]
            prom = round(ganancia / jugadas, 3) if jugadas > 0 else 0
            print(f"{c:<10} {jugadas:<10} {ganancia:<12.1f} {prom:<10}")
        print()

agente = AgenteConecta4()
print("Agente creado correctamente.")

Agente creado correctamente.


3. Oponente aleatorio para entrenamiento

In [ ]:
def oponente_aleatorio(tablero):
   
   
    validas = columnas_validas(tablero)
    return random.choice(validas) if validas else None

print("Oponente aleatorio listo.")

Oponente aleatorio listo.


4. Simulación de entrenamiento

In [ ]:
def simular_partida(agente, mostrar=False):
   
    tablero = nuevo_tablero()
    ultima_col_agente = None

    for turno in range(FILAS * COLS):
        if tablero_lleno(tablero):
            break

        if turno % 2 == 0:
            
            col = agente.elegir_columna(tablero)
            if col is None:
                break
            ultima_col_agente = col
            poner_ficha(tablero, col, AGENTE)
            if mostrar:
                print(f"[Agente juega en columna {col}]")
                mostrar_tablero(tablero)
            if hay_ganador(tablero, AGENTE):
                agente.actualizar(ultima_col_agente, +1)
                return 'agente'
            

        else:
            
            col = oponente_aleatorio(tablero)
            if col is None:
                break
            poner_ficha(tablero, col, HUMANO)
            if mostrar:
                print(f"[Oponente juega en columna {col}]")
                mostrar_tablero(tablero)
            if hay_ganador(tablero, HUMANO):
                if ultima_col_agente is not None:
                    agente.actualizar(ultima_col_agente, -1)
                return 'oponente'

    

    if ultima_col_agente is not None:
        agente.actualizar(ultima_col_agente, 0)
    return 'empate'



victorias_agente  = 0
victorias_humano  = 0
empates           = 0
PARTIDAS          = 500

for i in range(PARTIDAS):
    resultado = simular_partida(agente)
    if resultado == 'agente':
        victorias_agente += 1
    elif resultado == 'oponente':
        victorias_humano += 1
    else:
        empates += 1

print(f"Resultados tras {PARTIDAS} partidas de entrenamiento:")
print(f"  Agente ganó  : {victorias_agente}  ({victorias_agente/PARTIDAS*100:.1f}%)")
print(f"  Oponente ganó: {victorias_humano}  ({victorias_humano/PARTIDAS*100:.1f}%)")
print(f"  Empates      : {empates}  ({empates/PARTIDAS*100:.1f}%)")

agente.mostrar_estadisticas()

Resultados tras 500 partidas de entrenamiento:
  Agente ganó  : 483  (96.6%)
  Oponente ganó: 17  (3.4%)
  Empates      : 0  (0.0%)

--- Estadísticas del agente ---
Columna    Jugadas    Ganancia     Promedio  
---------------------------------------------
0          124        118.0        0.952     
1          119        105.0        0.882     
2          115        105.0        0.913     
3          142        138.0        0.972     



5. Ver cómo calcula UCB el agente

In [ ]:
tablero_prueba = nuevo_tablero()

print("Puntajes UCB para cada columna (tablero vacío):")
print(f"{'Columna':<10} {'UCB Score':<12}")
print("-" * 24)
for c in range(COLS):
    score = agente.ucb(c)
    score_str = f"{score:.4f}" if score != float('inf') else "inf (no explorada)"
    print(f"  {c:<8} {score_str}")

mejor = agente.elegir_columna(tablero_prueba)
print(f"\nEl agente elegiría la columna: {mejor}")

Puntajes UCB para cada columna (tablero vacío):
Columna    UCB Score   
------------------------
  0        1.2682
  1        1.2055
  2        1.2418
  3        1.2677

El agente elegiría la columna: 0


6. Partida ejemplo paso a paso

In [ ]:
print("="*40)
print(" PARTIDA DE EJEMPLO (Agente vs Aleatorio)")
print("="*40)
resultado_demo = simular_partida(agente, mostrar=True)
print(f"Resultado final: {'AGENTE GANA ✓' if resultado_demo == 'agente' else 'Oponente gana' if resultado_demo == 'oponente' else 'Empate'}")

 PARTIDA DE EJEMPLO (Agente vs Aleatorio)
[Agente juega en columna 0]

  col: 0   1   2   3
  -------------------
f0 | .  .  .  . |
f1 | .  .  .  . |
f2 | .  .  .  . |
f3 | .  .  .  . |
f4 | O  .  .  . |
  -------------------

[Oponente juega en columna 0]

  col: 0   1   2   3
  -------------------
f0 | .  .  .  . |
f1 | .  .  .  . |
f2 | .  .  .  . |
f3 | X  .  .  . |
f4 | O  .  .  . |
  -------------------

[Agente juega en columna 0]

  col: 0   1   2   3
  -------------------
f0 | .  .  .  . |
f1 | .  .  .  . |
f2 | O  .  .  . |
f3 | X  .  .  . |
f4 | O  .  .  . |
  -------------------

[Oponente juega en columna 3]

  col: 0   1   2   3
  -------------------
f0 | .  .  .  . |
f1 | .  .  .  . |
f2 | O  .  .  . |
f3 | X  .  .  . |
f4 | O  .  .  X |
  -------------------

[Agente juega en columna 0]

  col: 0   1   2   3
  -------------------
f0 | .  .  .  . |
f1 | O  .  .  . |
f2 | O  .  .  . |
f3 | X  .  .  . |
f4 | O  .  .  X |
  -------------------

[Oponente juega en columna 3]

7. modo interactivo – juega contra el agente

In [ ]:
def jugar_contra_agente():
    tablero = nuevo_tablero()
    print("=" * 38)
    print("  Conecta 4 simplificado – 3 en línea")
    print("  Tú eres X | Agente es O")
    print("=" * 38)
    mostrar_tablero(tablero)

    for turno in range(FILAS * COLS):
        if tablero_lleno(tablero):
            print("¡Empate! No quedan casillas.")
            break

        if turno % 2 == 0:
            # Turno del humano
            validas = columnas_validas(tablero)
            col = -1
            while col not in validas:
                try:
                    col = int(input(f"Tu turno. Elige columna {validas}: "))
                    if col not in validas:
                        print("Columna no válida, intenta de nuevo.")
                except ValueError:
                    print("Escribe un número.")
            poner_ficha(tablero, col, HUMANO)
            mostrar_tablero(tablero)
            if hay_ganador(tablero, HUMANO):
                print("¡Ganaste! Bien jugado.")
                break
        else:
            

            col = agente.elegir_columna(tablero)
            print(f"El agente juega en columna {col}.")
            poner_ficha(tablero, col, AGENTE)
            agente.actualizar(col, 0) 
            mostrar_tablero(tablero)
            if hay_ganador(tablero, AGENTE):
                print("El agente ganó esta vez. ¡Sigue intentando!")
                break




Para jugar contra el agente, quita el '#' de la última línea y ejecuta esta celda.


descripcion

se esta haciendo uso de un tablero 5x4 y el oponente es deconocido y las recompensa es de +1 si gana y -1 si pierde y 0 empata
con mas partidas de entrenamiento el agente identifica que columnas le dan mejores resultados y reduce la exploracion innecesaria concentrandose en las jugadas mas ganadoras
el agente no puede ver las intenciones del oponente, solo el estado visible del tablero el entorno cambia cada turno dependiendo de lo que juegue el agente y lo que juegue el oponente.
elegir. primero revisa si puede ganar directamente o si debe bloquear al oponente si no hay jugada urgente, usa el al
goritmo ucb para decidir.
función de accion
la acción consiste en elegir una columna (0 a 3) y colocar la ficha en la fila maas baja disponible de esa columna solo se pueden elegir columnas que no estén llenas.